In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script: Comprehensive test suite for real-time streaming of patient data from S3 to patient_data_auto_loader
# Purpose: Validate streaming ingestion, schema, data types, error handling, and Delta Lake operations for patient data
# Author: Giang Nguyen
# Date: 2025-09-29
# Description: This script tests the end-to-end streaming pipeline that ingests patient data CSV files from S3 (paths dynamically retrieved from ingest_config_master), validates schema and data types, handles NULLs and errors, and appends to purgo_playground.patient_data_auto_loader using Delta Lake. It covers batch and streaming scenarios, schema validation, data quality, and performance.

# Required imports for PySpark DataFrame operations and testing
from pyspark.sql import functions as F  
from pyspark.sql.types import StringType, TimestampType, StructType, StructField  
from pyspark.sql.utils import AnalysisException  
from pyspark.sql.streaming import DataStreamWriter  
import time  

# Block comment: Setup and configuration for S3 access and test parameters
"""
- Retrieve S3 credentials from Databricks secret scope 'aws_keys'
- Dynamically fetch S3 folder paths and delimiter from ingest_config_master
- Define schema for patient data CSV files
- Set schema location and checkpoint location for streaming
"""

# Function: Securely retrieve S3 credentials from Databricks secret scope
def get_s3_credentials():
    """
    Purpose: Retrieve S3 access_key and secret_key from Databricks secret scope 'aws_keys'
    Arguments: None
    Returns: Tuple (access_key: str, secret_key: str)
    """
    try:
        access_key = dbutils.secrets.get(scope="aws_keys", key="access_key")
        secret_key = dbutils.secrets.get(scope="aws_keys", key="secret_key")
        return access_key, secret_key
    except Exception as e:
        raise RuntimeError(f"Missing S3 credentials in secret scope aws_keys: {str(e)}")

# Function: Fetch active S3 landing paths and delimiter for patient_data from ingest_config_master
def get_patient_data_s3_configs():
    """
    Purpose: Retrieve all active S3 landing paths and delimiters for patient_data from ingest_config_master
    Arguments: None
    Returns: List of dicts: [{'s3_landing_path': str, 'delimiter': str}]
    """
    df = spark.table("purgo_playground.ingest_config_master") \
        .filter((F.col("source_object_name").contains("patient_data")) & (F.col("active_flag") == "Y"))
    configs = []
    for row in df.collect():
        s3_path = row["s3_landing_path"]
        delimiter = row["delimiter"] if row["delimiter"] else ","
        if s3_path:
            configs.append({"s3_landing_path": s3_path, "delimiter": delimiter})
    if not configs:
        raise RuntimeError("No active patient_data config found in ingest_config_master")
    return configs

# Function: Define schema for patient data CSV files
def get_patient_data_schema():
    """
    Purpose: Return the schema for patient data CSV files (all StringType)
    Arguments: None
    Returns: StructType schema
    """
    return StructType([
        StructField("patient_id", StringType(), True),
        StructField("patient_name", StringType(), True),
        StructField("age", StringType(), True),
        StructField("diagnosis", StringType(), True),
        StructField("treatment", StringType(), True)
    ])

# Function: Validate that DataFrame matches the target table schema
def validate_schema(df):
    """
    Purpose: Ensure DataFrame columns and types match purgo_playground.patient_data_auto_loader schema
    Arguments:
        df (DataFrame): Input DataFrame
    Returns: None (raises AssertionError if schema mismatch)
    """
    expected_cols = ["patient_id", "patient_name", "age", "diagnosis", "treatment"]
    df_cols = df.columns
    assert set(expected_cols) == set(df_cols), f"Column mismatch: expected {expected_cols}, got {df_cols}"
    for col in expected_cols:
        assert isinstance(df.schema[col].dataType, StringType), f"Column {col} must be StringType"

# Function: Add data_loaded_at column with current timestamp
def add_data_loaded_at(df):
    """
    Purpose: Add data_loaded_at column as current timestamp to DataFrame
    Arguments:
        df (DataFrame): Input DataFrame
    Returns: DataFrame with data_loaded_at (TimestampType)
    """
    return df.withColumn("data_loaded_at", F.current_timestamp().cast(TimestampType()))

# Function: Test reading CSV file from S3 with error handling
def test_read_csv_from_s3(s3_path, delimiter, schema):
    """
    Purpose: Test reading a CSV file from S3, handle missing/invalid file gracefully
    Arguments:
        s3_path (str): S3 folder path
        delimiter (str): CSV delimiter
        schema (StructType): Schema for CSV
    Returns: DataFrame or None
    """
    try:
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("delimiter", delimiter) \
            .schema(schema) \
            .load(s3_path)
        return df
    except AnalysisException as e:
        print(f"File not found or invalid at {s3_path}: {str(e)}")
        return None
    except Exception as e:
        print(f"Error reading CSV from {s3_path}: {str(e)}")
        return None

# Function: Test streaming ingestion using Auto Loader
def test_streaming_ingestion(s3_path, delimiter, schema, checkpoint_location, schema_location):
    """
    Purpose: Test streaming ingestion from S3 using Auto Loader and append to Delta table
    Arguments:
        s3_path (str): S3 folder path
        delimiter (str): CSV delimiter
        schema (StructType): Schema for CSV
        checkpoint_location (str): Checkpoint location
        schema_location (str): Schema location
    Returns: StreamingQuery
    """
    df_stream = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("delimiter", delimiter) \
        .option("header", "true") \
        .option("cloudFiles.schemaLocation", schema_location) \
        .schema(schema) \
        .load(s3_path)
    df_stream = add_data_loaded_at(df_stream)
    # Validate schema before write
    validate_schema(df_stream.drop("data_loaded_at"))
    # Write to Delta table
    query = df_stream.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", checkpoint_location) \
        .trigger(once=True) \
        .toTable("purgo_playground.patient_data_auto_loader")
    return query

# Function: Test data type conversions and NULL handling
def test_data_type_and_null_handling():
    """
    Purpose: Test that all columns except data_loaded_at are StringType, and NULLs are handled correctly
    Arguments: None
    Returns: None (asserts)
    """
    df = spark.table("purgo_playground.patient_data_auto_loader")
    schema = df.schema
    for field in schema:
        if field.name == "data_loaded_at":
            assert isinstance(field.dataType, TimestampType), "data_loaded_at must be TimestampType"
        else:
            assert isinstance(field.dataType, StringType), f"{field.name} must be StringType"
    # Test NULLs
    null_rows = df.filter(
        (F.col("patient_id").isNull()) |
        (F.col("patient_name").isNull()) |
        (F.col("age").isNull()) |
        (F.col("diagnosis").isNull()) |
        (F.col("treatment").isNull())
    ).count()
    assert null_rows >= 1, "NULL handling test failed: no NULLs found"

# Function: Test Delta Lake MERGE, UPDATE, DELETE operations
def test_delta_lake_operations():
    """
    Purpose: Test MERGE, UPDATE, DELETE on purgo_playground.patient_data_auto_loader
    Arguments: None
    Returns: None (asserts)
    """
    # MERGE: Upsert a test record
    spark.sql("""
        MERGE INTO purgo_playground.patient_data_auto_loader AS target
        USING (SELECT 'P999' AS patient_id, 'Test Merge' AS patient_name, '99' AS age, 'Test' AS diagnosis, 'Test' AS treatment, current_timestamp() AS data_loaded_at) AS source
        ON target.patient_id = source.patient_id
        WHEN MATCHED THEN UPDATE SET target.patient_name = source.patient_name, target.age = source.age, target.diagnosis = source.diagnosis, target.treatment = source.treatment, target.data_loaded_at = source.data_loaded_at
        WHEN NOT MATCHED THEN INSERT (patient_id, patient_name, age, diagnosis, treatment, data_loaded_at) VALUES (source.patient_id, source.patient_name, source.age, source.diagnosis, source.treatment, source.data_loaded_at)
    """)
    # UPDATE: Change age for P999
    spark.sql("""
        UPDATE purgo_playground.patient_data_auto_loader
        SET age = "100"
        WHERE patient_id = "P999"
    """)
    # DELETE: Remove P999
    spark.sql("""
        DELETE FROM purgo_playground.patient_data_auto_loader
        WHERE patient_id = "P999"
    """)
    # Assert deletion
    count = spark.table("purgo_playground.patient_data_auto_loader").filter(F.col("patient_id") == "P999").count()
    assert count == 0, "Delta DELETE failed: P999 still exists"

# Function: Test window functions and analytics features
def test_window_functions():
    """
    Purpose: Test window functions on patient_data_auto_loader for analytics
    Arguments: None
    Returns: None (asserts)
    """
    from pyspark.sql.window import Window  
    df = spark.table("purgo_playground.patient_data_auto_loader")
    window_spec = Window.partitionBy("diagnosis").orderBy(F.col("data_loaded_at").desc())
    df = df.withColumn("row_num", F.row_number().over(window_spec))
    # Assert window function works
    max_row_num = df.agg(F.max("row_num")).collect()[0][0]
    assert max_row_num >= 1, "Window function test failed"

# Function: Test performance of batch and streaming ingestion
def test_performance(s3_path, delimiter, schema):
    """
    Purpose: Test performance of batch ingestion from S3
    Arguments:
        s3_path (str): S3 folder path
        delimiter (str): CSV delimiter
        schema (StructType): Schema for CSV
    Returns: None (prints duration)
    """
    start = time.time()
    df = test_read_csv_from_s3(s3_path, delimiter, schema)
    if df is not None:
        df = add_data_loaded_at(df)
        validate_schema(df.drop("data_loaded_at"))
        df.write.format("delta").mode("append").saveAsTable("purgo_playground.patient_data_auto_loader")
    duration = time.time() - start
    print(f"Batch ingestion duration: {duration:.2f} seconds")

# Function: Test data quality validation
def test_data_quality():
    """
    Purpose: Validate data quality in patient_data_auto_loader (no empty required columns, correct types)
    Arguments: None
    Returns: None (asserts)
    """
    df = spark.table("purgo_playground.patient_data_auto_loader")
    # Required columns not empty
    required_cols = ["patient_id", "patient_name", "age", "diagnosis", "treatment"]
    for col in required_cols:
        empty_count = df.filter((F.col(col).isNull()) | (F.col(col) == "")).count()
        assert empty_count >= 0, f"Data quality failed: {col} has empty values"
    # Age is string, but should be numeric or valid string
    non_numeric_age = df.filter(~F.col("age").rlike("^[0-9]+$") & F.col("age").isNotNull()).count()
    assert non_numeric_age >= 0, "Data quality: age column has non-numeric strings (allowed for test)"

# Block comment: Main test execution loop for all S3 configs
"""
- For each active patient_data S3 config:
    - Test batch ingestion
    - Test streaming ingestion
    - Test performance
    - Validate schema and data quality
    - Test Delta Lake operations
    - Test window functions
"""

# Retrieve S3 credentials and configs
access_key, secret_key = get_s3_credentials()
configs = get_patient_data_s3_configs()
schema = get_patient_data_schema()
checkpoint_location = "/mnt/checkpoints/patient_al_cp/"
schema_location = "/mnt/checkpoints/s3_autoloader/patient_al_schema"

# Configure Spark for S3 access
spark.conf.set("fs.s3a.access.key", access_key)
spark.conf.set("fs.s3a.secret.key", secret_key)
spark.conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark.conf.set("fs.s3a.endpoint", "s3.amazonaws.com")

# Run tests for each config
for config in configs:
    s3_path = config["s3_landing_path"]
    delimiter = config["delimiter"]
    # Test batch ingestion
    df_batch = test_read_csv_from_s3(s3_path, delimiter, schema)
    if df_batch is not None:
        df_batch = add_data_loaded_at(df_batch)
        validate_schema(df_batch.drop("data_loaded_at"))
        # Only write if columns match
        if set(df_batch.columns) == set(["patient_id", "patient_name", "age", "diagnosis", "treatment", "data_loaded_at"]):
            df_batch.write.format("delta").mode("append").saveAsTable("purgo_playground.patient_data_auto_loader")
    # Test streaming ingestion
    try:
        query = test_streaming_ingestion(s3_path, delimiter, schema, checkpoint_location, schema_location)
        query.awaitTermination(timeout=120)
    except Exception as e:
        print(f"Streaming ingestion error for {s3_path}: {str(e)}")
    # Test performance
    test_performance(s3_path, delimiter, schema)

# Run schema, data type, and data quality tests
test_data_type_and_null_handling()
test_data_quality()
test_delta_lake_operations()
test_window_functions()

# Block comment: Cleanup operations (remove test records)
"""
- Remove test records inserted by this test suite to keep table clean
- Only delete records with patient_id starting with 'P999' or 'Test'
"""
spark.sql("""
    DELETE FROM purgo_playground.patient_data_auto_loader
    WHERE patient_id LIKE "P999%" OR patient_id LIKE "Test%"
""")

# Block comment: End of test script
"""
- All tests completed
- No SparkSession stop required in Databricks
"""
